#### process for windows
1. create empty arrays for each column (width, height if etc)
2. iterate over all model.Objects:
    - filter out only the objects whose names are "window"
    - extract relevant parameters (dimensions) and append them into respective arrays
        - height is easy (always z axis)
        - width and depth need comparison (assume depth is always < width)
        - orientation is tricky, so in the for loop we save x and y information as well as W-E / N-S orientation
        - we also save z information for floor calculation
        
3. create dataframe out of all the arrays
4. One way to differentiate north from south / east from west is to look at x and y values
    - if we have x_mean for all the W-E windows, then x>x_mean is East-facing, and x<x_mean is West-facing. Vice versa for N-S.
5. Manipulate dataframe to create orientation column

#### process for floors
1. iterate over all model.Objects:
    - filter out only the objects whose names are "floor"
    - get their z values

2. go over each window's z value, compare against floors' z values and determine the floor of the window.

#### process for indexing the windows
1. determine format
    - "F1N15" means the 15th window on floor 1, and it faces North
2. find our how many windows are on each floor
3. build the index


In [1]:
import pandas as pd
import numpy as np
import rhino3dm as rh

In [2]:
model = rh.File3dm.Read("Building_model_5floor.3dm")

In [3]:
width_s =[]
height_s = []
depth_s = []
orientation_s = []
x_s = []
y_s = []
z_s = []


for obj in model.Objects:
    name = obj.Attributes.Name
    if name == "window":
        geometry = obj.Geometry
        bbox = geometry.GetBoundingBox()
        diagonal = bbox.Diagonal
        height_s.append(diagonal.Z)
        
        x_s.append(bbox.Min.X)
        y_s.append(bbox.Min.Y)
        z_s.append(bbox.Min.Z)
        if diagonal.X< diagonal.Y:
            width_s.append(diagonal.Y)
            depth_s.append(diagonal.X)
            orientation_s.append("WE")
        else:
            width_s.append(diagonal.X)
            depth_s.append(diagonal.Y)
            orientation_s.append("NS")
            
df = pd.DataFrame(list(zip(width_s, height_s, depth_s, orientation_s, x_s, y_s, z_s)),
                  columns =['width','height','depth','orientation_group','x','y','z'])
df['area'] = df.width * df.height                    

In [4]:
y_mean =  np.mean(list(set(df[df['orientation_group']=='NS'].y)))
x_mean =  np.mean(list(set(df[df['orientation_group']=='WE'].x)))

def find_orientation(row):
    if row.orientation_group =='NS':
        if row.y> y_mean:
            return "N"
        else:
            return "S"
    if row.orientation_group =='WE':
        if row.x> x_mean:
            return "E"
        else:
            return "W"

df['orientation'] = df.apply(find_orientation, axis=1)
        

In [5]:
floor_z_s = []
for obj in model.Objects:
    name = obj.Attributes.Name
    if name =='floor':
        floor_z_s.append(obj.Geometry.GetBoundingBox().Min.Z)
        
floor_z_s.sort()

def map_z_to_floor(row):
    for f in range(5):
        if row.z > floor_z_s[f] and row.z < floor_z_s[f+1]:
            return f

df['floor'] = df.apply(map_z_to_floor, axis =1)

In [10]:
df.sort_values(['floor','orientation'], inplace=True)
idx_s  = []
cnt_s =[0,0,0,0,0]
for f in df.floor:
    idx_s.append(cnt_s[f]+1)
    cnt_s[f]+=1
df['idx'] = idx_s
df['name']= df.apply(lambda row: 'F'+str(row.floor)+ row.orientation+str(row.idx),axis=1)

df.reset_index(drop=True, inplace=True)
df['index'] = df.index + 1

df = df[['index','name','floor','orientation','width','height','depth','area']]

## for any column whose value is of type float, round it to 2 decimal places
for c in df.columns:
    if isinstance(df.loc[0,c], float):
        df[c] = df[c].round(2)
df

/usr/local/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


,index,name,floor,orientation,width,height,depth,area
0,1,F0E1,0,E,1.46,2.02,0.17,2.94
1,2,F0E2,0,E,1.46,2.02,0.17,2.94
2,3,F0E3,0,E,8.01,2.02,0.17,16.17
3,4,F0N4,0,N,2.29,4.05,0.17,9.26
4,5,F0N5,0,N,2.00,4.05,0.17,8.11
...,...,...,...,...,...,...,...,...
90,91,F4S15,4,S,3.14,4.05,0.17,12.71
91,92,F4W16,4,W,1.77,2.58,0.17,4.57
92,93,F4W17,4,W,3.16,2.58,0.17,8.16
93,94,F4W18,4,W,1.77,2.58,0.17,4.57


In [12]:
df.to_csv('windows_information.csv')